# MPI-Matrixmultiplikation direkt aus dem Notebook

Dieses Notebook zeigt eine **parallele Matrix-Vektor-Multiplikation** mit `mpi4py`.

Wir berechnen:

$$
y = A \cdot x
$$

Dabei ist:

- `A` eine quadratische Matrix
- `x` ein Vektor
- `y` der Ergebnisvektor

Die Idee der Parallelisierung ist einfach:

1. Prozess **0** erzeugt Matrix und Vektor.
2. Der Vektor `x` wird an **alle Prozesse** verteilt.
3. Die Matrix `A` wird in **Zeilenblöcke** zerlegt.
4. Jeder Prozess berechnet seinen **lokalen Anteil** des Ergebnisvektors.
5. Am Ende werden alle Teilresultate wieder bei Prozess **0** eingesammelt.

So sieht man sehr schön den Einsatz von:
- `bcast`
- `scatter`
- lokaler Berechnung
- `gather`


## Schritt 1: MPI-Programm als Datei schreiben

Da MPI nicht einfach „in derselben Notebook-Zelle parallel losläuft“, schreiben wir zuerst ein normales Python-Skript.

Dieses Skript wird danach mit `mpiexec` gestartet.


In [ ]:
code = r'''from mpi4py import MPI
import numpy as np

# --------------------------------------------------
# MPI-Grundlagen
# --------------------------------------------------
# COMM_WORLD umfasst alle gestarteten Prozesse.
comm = MPI.COMM_WORLD

# rank = eindeutige ID des Prozesses
rank = comm.Get_rank()

# size = Gesamtzahl aller Prozesse
size = comm.Get_size()

# --------------------------------------------------
# Problemgröße
# --------------------------------------------------
# Wir verwenden eine quadratische Matrix A der Größe N x N
# und einen Vektor x der Länge N.
N = 4

# --------------------------------------------------
# Schritt 1: Daten nur auf rank 0 erzeugen
# --------------------------------------------------
# Nur Prozess 0 erzeugt die Eingabedaten.
# Alle anderen Prozesse setzen A und x zunächst auf None.
if rank == 0:
    np.random.seed(42)  # für reproduzierbare Ergebnisse
    A = np.random.randint(1, 10, (N, N))
    x = np.random.randint(1, 10, N)

    print(f"[rank {rank}] Matrix A:\n{A}\n")
    print(f"[rank {rank}] Vektor x:\n{x}\n")
else:
    A = None
    x = None

# --------------------------------------------------
# Schritt 2: Broadcast des Vektors x
# --------------------------------------------------
# Alle Prozesse benötigen den kompletten Vektor x,
# weil jede lokale Zeile von A mit demselben x multipliziert wird.
x = comm.bcast(x, root=0)

print(f"[rank {rank}] hat nach bcast den Vektor x = {x}")

# --------------------------------------------------
# Schritt 3: Matrix A in Zeilenblöcke zerlegen
# --------------------------------------------------
# Nur rank 0 teilt A in Teilmatrizen auf.
# np.array_split sorgt dafür, dass auch ungerade Verteilungen funktionieren.
if rank == 0:
    split_A = np.array_split(A, size, axis=0)

    print(f"\n[rank {rank}] Aufteilung von A in {size} Blöcke:")
    for i, block in enumerate(split_A):
        print(f"Block für Prozess {i}:\n{block}\n")
else:
    split_A = None

# --------------------------------------------------
# Schritt 4: Scatter
# --------------------------------------------------
# Jeder Prozess erhält genau einen Block seiner Matrixzeilen.
local_A = comm.scatter(split_A, root=0)

print(f"[rank {rank}] hat lokalen Matrixblock:\n{local_A}\n")

# --------------------------------------------------
# Schritt 5: Lokale Berechnung
# --------------------------------------------------
# Jeder Prozess berechnet nur seinen eigenen Anteil am Ergebnisvektor.
local_y = np.dot(local_A, x)

print(f"[rank {rank}] berechnet lokalen Ergebnisblock y_local = {local_y}")

# --------------------------------------------------
# Schritt 6: Gather
# --------------------------------------------------
# Alle lokalen Teilvektoren werden wieder bei rank 0 eingesammelt.
gathered_y = comm.gather(local_y, root=0)

# --------------------------------------------------
# Schritt 7: Endergebnis auf rank 0 zusammensetzen
# --------------------------------------------------
if rank == 0:
    y = np.concatenate(gathered_y)

    print(f"\n[rank {rank}] eingesammelte Teilvektoren:")
    for i, part in enumerate(gathered_y):
        print(f"Von Prozess {i}: {part}")

    print(f"\n[rank {rank}] Ergebnisvektor y:\n{y}\n")

    # Optionaler Korrektheitscheck mit NumPy
    y_check = A.dot(x)
    print(f"[rank {rank}] Kontrolle mit NumPy A.dot(x):\n{y_check}\n")
    print(f"[rank {rank}] Stimmen beide Ergebnisse überein? {np.array_equal(y, y_check)}")
'''

with open("mpi_matrixmultiplikation_exec.py", "w", encoding="utf-8") as f:
    f.write(code)

print("Datei mpi_matrixmultiplikation_exec.py wurde erstellt.")

## Schritt 2: MPI-Programm aus dem Notebook heraus starten

Hier verwenden wir:

- den absoluten Pfad `/usr/bin/mpiexec`
- `--mca plm isolated`, damit OpenMPI in dieser Umgebung keinen SSH-Start versucht
- `sys.executable`, damit exakt derselbe Python-Interpreter wie im Notebook benutzt wird

Zusätzlich filtern wir die bekannte X11-Warnung aus `stderr` heraus.


In [ ]:
import os
import sys
import subprocess

mpiexec = "/usr/bin/mpiexec"
env = os.environ.copy()

# Falls das Notebook als root läuft (typisch in manchen Containern)
if hasattr(os, "geteuid") and os.geteuid() == 0:
    env["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
    env["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"

cmd = [
    mpiexec,
    "--mca", "plm", "isolated",
    "-n", "4",
    sys.executable,
    "mpi_matrixmultiplikation_exec.py",
]

print("Starte Befehl:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env=env
)

print("\nSTDOUT:\n")
print(result.stdout if result.stdout else "(leer)")

filtered_stderr = "\n".join(
    line for line in result.stderr.splitlines()
    if "Authorization required, but no authorization protocol specified" not in line
).strip()

if filtered_stderr:
    print("\nWeitere STDERR-Meldungen:\n")
    print(filtered_stderr)

print("\nReturncode:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("MPI-Ausführung fehlgeschlagen.")

## Erklärung der MPI-Operationen in diesem Beispiel

### `comm.bcast(x, root=0)`
Prozess 0 besitzt den Vektor `x` und verteilt ihn an alle anderen Prozesse.

Warum ist das nötig?  
Jeder Prozess berechnet später:

$$
y_i = A_i \cdot x
$$

und braucht deshalb den **kompletten Vektor `x`**.

---

### `comm.scatter(split_A, root=0)`
Prozess 0 zerlegt die Matrix `A` in mehrere Zeilenblöcke und verteilt diese auf die Prozesse.

Beispiel bei 4 Prozessen:

- Prozess 0 bekommt die ersten Zeilen
- Prozess 1 die nächsten
- usw.

So arbeitet jeder Prozess nur auf **seinem eigenen Teilproblem**.

---

### `np.dot(local_A, x)`
Das ist die eigentliche lokale Rechnung.

Jeder Prozess multipliziert **seinen Matrixblock** mit dem vollständigen Vektor `x`.

Dadurch entsteht ein **lokaler Teil des Ergebnisvektors**.

---

### `comm.gather(local_y, root=0)`
Alle lokalen Ergebnisstücke werden wieder zu Prozess 0 zurückgeschickt.

Dort kann Prozess 0 die Teilvektoren in der richtigen Reihenfolge zusammensetzen.

---

### `np.concatenate(gathered_y)`
Das ist keine MPI-Operation, sondern normales NumPy.

Hier werden die eingesammelten Teilvektoren zu einem einzigen Ergebnisvektor `y` zusammengesetzt.


## Einordnung

An diesem Beispiel sieht man mehrere Grundideen verteilter bzw. paralleler Systeme:

### 1. Datenverteilung
Nicht jeder Prozess bekommt alle Daten vollständig.  
Die Matrix wird aufgeteilt, der Vektor wird vervielfältigt.

### 2. Datenlokalität
Jeder Prozess arbeitet nur auf seinem lokalen Block.

### 3. Kommunikation + Berechnung
Das Programm besteht aus beidem:
- **Kommunikation**: `bcast`, `scatter`, `gather`
- **Berechnung**: `np.dot(...)`

### 4. Sammeln des Gesamtergebnisses
Teilresultate müssen wieder korrekt zusammengesetzt werden.

### 5. Lastverteilung
Durch `np.array_split(...)` funktioniert das Beispiel auch dann, wenn die Zeilenzahl nicht perfekt durch die Anzahl der Prozesse teilbar ist.


## Fragen zum Nachdenken:

- Warum wird der Vektor `x` an alle Prozesse verteilt, die Matrix `A` aber nicht?
- Warum eignet sich die Matrix-Vektor-Multiplikation gut für Parallelisierung?
- Welche Kommunikation ist notwendig, welche nicht?
- Was passiert, wenn wir mehr Prozesse als Matrixzeilen haben?
- Wo entstehen Overheads durch Kommunikation?


## Kurzfazit

Dieses Notebook zeigt ein typisches Muster der parallelen Programmierung mit MPI:

- **Daten erzeugen**
- **Daten verteilen**
- **lokal rechnen**
- **Ergebnisse einsammeln**
- **Gesamtergebnis rekonstruieren**

Damit ist es ein sehr gutes Einstiegsbeispiel für `mpi4py`.
